# Hybrid YOLOv9-DETR Strawberry Disease Detection - Quick Demo

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AmirmasoudGhorbani/hybrid-yolov9-detr-strawberry-disease/blob/main/notebooks/demo.ipynb)

This notebook demonstrates the hybrid detection pipeline on sample images.
It loads pre-trained YOLOv9 and DETR weights, runs both models, fuses their
predictions via IoU-based box selection, and visualises the results.

**No training required** - just provide your model weights and an image.

## 1. Setup

In [ ]:
# Clone the repo (skip if running locally)
!git clone https://github.com/AmirmasoudGhorbani/hybrid-yolov9-detr-strawberry-disease.git
%cd hybrid-yolov9-detr-strawberry-disease
!pip install -q -r requirements.txt

In [ ]:
import os
import torch
import matplotlib.pyplot as plt
from PIL import Image

print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 2. Configure Paths

Point these at your trained model weights. If you're using Google Drive in Colab,
mount it first and update the paths.

In [ ]:
# Uncomment to mount Google Drive in Colab
# from google.colab import drive
# drive.mount('/content/drive')

# Update these paths to point at your trained weights
YOLO_WEIGHTS = "/content/drive/MyDrive/yolov9_lastfinetune_results/A100_finetune_best_resume/weights/best.pt"
DETR_MODEL = "/content/drive/MyDrive/detr_strawberry_model_extrafinetuned"
DETR_PROCESSOR = "/content/drive/MyDrive/detr_strawberry_processor_extrafinetuned"

# Path to a test image
TEST_IMAGE = "/content/drive/MyDrive/dataset/dataset json format/test/your_image.jpg"

## 3. Load Models

In [ ]:
from src.predict import load_models, run_hybrid_inference, draw_predictions
from src.config import DISEASE_NAMES

yolo_model, detr_model, processor = load_models(
    yolo_weights=YOLO_WEIGHTS,
    detr_model_path=DETR_MODEL,
    detr_processor_path=DETR_PROCESSOR,
)
print("Models loaded successfully!")

## 4. Run Hybrid Inference

The pipeline:
1. **YOLOv9** proposes bounding boxes (fast, high recall)
2. **DETR** refines with transformer attention (better localisation)
3. **IoU-based fusion** keeps the best box from each model

In [ ]:
image = Image.open(TEST_IMAGE).convert("RGB")

results = run_hybrid_inference(
    image, yolo_model, detr_model, processor,
    threshold=0.5,
    iou_threshold=0.5,
)

print(f"Detections: {len(results['boxes'])}")
for box, label, score, source in zip(
    results['boxes'], results['labels'], results['scores'], results['sources']
):
    name = DISEASE_NAMES[int(label)] if int(label) < len(DISEASE_NAMES) else f"class_{int(label)}"
    print(f"  {name}: {score:.1%} (from {source.upper()})")

## 5. Visualise: YOLO vs DETR vs Hybrid

In [ ]:
fig = draw_predictions(image, results, show_comparison=True)
plt.show()

## 6. Try Your Own Images

Upload an image or change the path below to try the hybrid model on different inputs.

In [ ]:
# Upload from local machine (Colab only)
# from google.colab import files
# uploaded = files.upload()
# TEST_IMAGE = list(uploaded.keys())[0]

# Or specify a path directly:
# TEST_IMAGE = "path/to/your/strawberry_image.jpg"

image = Image.open(TEST_IMAGE).convert("RGB")
results = run_hybrid_inference(image, yolo_model, detr_model, processor)
fig = draw_predictions(image, results, show_comparison=True)
plt.show()

## Disease Classes

The model detects 12 classes of strawberry diseases and ripeness states:

| # | Class | Type |
|---|-------|------|
| 0 | Angular Leafspot | Disease |
| 1 | Anthracnose Fruit Rot | Disease |
| 2 | Early-Turning | Ripeness |
| 3 | Gray Mold | Disease |
| 4 | Green-Strawberry | Ripeness |
| 5 | Late-Turning | Ripeness |
| 6 | Leaf Spot | Disease |
| 7 | Powdery Mildew Fruit | Disease |
| 8 | Powdery Mildew Leaf | Disease |
| 9 | Red-Turning | Ripeness |
| 10 | Turning | Ripeness |
| 11 | White-Strawberry | Ripeness |